# Project 3 — Morgan Fingerprint QSAR Regression

This notebook predicts solubility using Morgan fingerprints rather than manually selected descriptors. A Morgan fingerprint converts each molecule into a fixed-length binary vector based on circular atom environments.

Compared with descriptor QSAR, fingerprints are less directly interpretable but often capture richer substructure information. This makes them useful for similarity search and property prediction.

The notebook uses the same public-data strategy as Project 2: load public data when available, fall back to the small built-in table when offline and cap the working dataset to 500 molecules.

## 1. Setup

Morgan fingerprints are generated with RDKit and converted into NumPy arrays for scikit-learn models. The generator controls fingerprint radius and vector size.

Radius controls how far atom environments extend. Fingerprint size controls how many binary positions are available to store those environments.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from rdkit import Chem, DataStructs
from rdkit.Chem import Draw
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load and clean regression data

The same solubility data strategy is used here so descriptor and fingerprint models can be compared on similar data. Public data gives a more realistic training set, while the fallback data protects offline execution.

Cleaning remains essential because fingerprints still require valid RDKit molecule objects.

In [ ]:
def make_demo_solubility_data():
    records = [
        {"name": "methanol", "smiles": "CO", "target_logS": 0.10},
        {"name": "ethanol", "smiles": "CCO", "target_logS": -0.20},
        {"name": "propanol", "smiles": "CCCO", "target_logS": -0.70},
        {"name": "butanol", "smiles": "CCCCO", "target_logS": -1.10},
        {"name": "pentanol", "smiles": "CCCCCO", "target_logS": -1.70},
        {"name": "hexanol", "smiles": "CCCCCCO", "target_logS": -2.20},
        {"name": "acetone", "smiles": "CC(=O)C", "target_logS": -0.20},
        {"name": "acetic acid", "smiles": "CC(=O)O", "target_logS": 0.10},
        {"name": "ethyl acetate", "smiles": "CCOC(=O)C", "target_logS": -0.70},
        {"name": "diethyl ether", "smiles": "CCOCC", "target_logS": -1.00},
        {"name": "benzene", "smiles": "c1ccccc1", "target_logS": -2.10},
        {"name": "toluene", "smiles": "Cc1ccccc1", "target_logS": -2.70},
        {"name": "phenol", "smiles": "Oc1ccccc1", "target_logS": -1.50},
        {"name": "aniline", "smiles": "Nc1ccccc1", "target_logS": -1.30},
        {"name": "benzoic acid", "smiles": "O=C(O)c1ccccc1", "target_logS": -1.80},
        {"name": "benzaldehyde", "smiles": "O=Cc1ccccc1", "target_logS": -1.90},
        {"name": "benzyl alcohol", "smiles": "OCc1ccccc1", "target_logS": -1.10},
        {"name": "vanillin", "smiles": "COc1cc(C=O)ccc1O", "target_logS": -1.20},
        {"name": "cinnamaldehyde", "smiles": "O=CC=Cc1ccccc1", "target_logS": -2.30},
        {"name": "limonene", "smiles": "CC1=CCC(CC1)C(=C)C", "target_logS": -4.30},
        {"name": "linalool", "smiles": "CC(C)=CCCC(C)(O)C=C", "target_logS": -2.80},
        {"name": "menthol", "smiles": "CC(C)C1CCC(C)CC1O", "target_logS": -3.20},
        {"name": "aspirin", "smiles": "CC(=O)OC1=CC=CC=C1C(=O)O", "target_logS": -2.30},
        {"name": "caffeine", "smiles": "Cn1cnc2c1c(=O)n(C)c(=O)n2C", "target_logS": -1.00},
        {"name": "paracetamol", "smiles": "CC(=O)NC1=CC=C(O)C=C1", "target_logS": -1.40},
        {"name": "ibuprofen", "smiles": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O", "target_logS": -4.00},
        {"name": "naphthalene", "smiles": "c1ccc2ccccc2c1", "target_logS": -3.60},
        {"name": "anthracene", "smiles": "c1ccc2cc3ccccc3cc2c1", "target_logS": -5.10},
        {"name": "glucose", "smiles": "C(C1C(C(C(C(O1)O)O)O)O)O", "target_logS": 0.70},
        {"name": "cholesterol", "smiles": "CC(C)CCCC(C)C1CCC2C3CCC4=CC(O)CCC4(C)C3CCC12C", "target_logS": -7.00},
    ]
    return pd.DataFrame(records)


def standardise_solubility_table(raw_df):
    column_lookup = {col.lower(): col for col in raw_df.columns}

    smiles_col = column_lookup.get("smiles")
    target_col = column_lookup.get("solubility") or column_lookup.get("target_logs") or column_lookup.get("logs")
    name_col = column_lookup.get("name")

    if smiles_col is None or target_col is None:
        raise ValueError("The public dataset does not contain the expected SMILES and solubility columns.")

    names = raw_df[name_col] if name_col is not None else [f"molecule_{i}" for i in range(len(raw_df))]

    # Keep only the columns needed by the downstream notebook.
    df = pd.DataFrame({
        "name": names,
        "smiles": raw_df[smiles_col],
        "target_logS": raw_df[target_col],
    })

    # Remove incomplete records before modelling.
    df = df.dropna(subset=["smiles", "target_logS"]).copy()

    # Convert the target to numeric LogS values.
    df["target_logS"] = pd.to_numeric(df["target_logS"], errors="coerce")
    df = df.dropna(subset=["target_logS"]).copy()

    return df


def load_aqsol_public_data(timeout_seconds=10):
    from urllib.request import Request, urlopen

    url = "https://raw.githubusercontent.com/mcsorkun/AqSolDB/master/results/data_curated.csv"
    request = Request(url, headers={"User-Agent": "chem-ml-practice-notebook"})

    # Use a network timeout so offline notebooks fall back instead of hanging.
    with urlopen(request, timeout=timeout_seconds) as response:
        raw_df = pd.read_csv(response, low_memory=False)

    return standardise_solubility_table(raw_df)


def load_solubility_data(max_molecules=500, random_state=42):
    try:
        df = load_aqsol_public_data()
        data_source = "AqSolDB public dataset"
    except Exception as error:
        print("Public dataset loading failed:", error)
        df = make_demo_solubility_data()
        data_source = "built-in demo dataset"

    # Randomly cap the dataset before descriptor/fingerprint calculation.
    if len(df) > max_molecules:
        df = df.sample(n=max_molecules, random_state=random_state).reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)

    return df, data_source

MAX_MOLECULES = 500
RANDOM_STATE = 42

df, data_source = load_solubility_data(max_molecules=MAX_MOLECULES, random_state=RANDOM_STATE)

# Convert SMILES strings into RDKit molecules after the 500-row cap.
df["mol"] = df["smiles"].apply(lambda smiles: Chem.MolFromSmiles(str(smiles)))

# Keep molecules that RDKit can parse.
df = df[df["mol"].notnull()].copy()

# Canonical SMILES helps identify repeated structures.
df["canonical_smiles"] = df["mol"].apply(lambda mol: Chem.MolToSmiles(mol, canonical=True))
df = df.drop_duplicates(subset=["canonical_smiles"]).reset_index(drop=True)

print("Data source:", data_source)
print("Clean dataset used:", df.shape)
df[["name", "canonical_smiles", "target_logS"]].head()


## 3. Generate Morgan fingerprints

Each molecule is converted into a binary vector. A bit is active when RDKit detects a circular atom environment that hashes to that bit position.

The output is machine-learning friendly because every molecule has the same vector length. The trade-off is that individual bits are less interpretable than simple descriptors.

In [ ]:
FP_SIZE = 2048

# Radius 2 corresponds to atom environments up to two bonds away.
morgan_generator = GetMorganGenerator(radius=2, fpSize=FP_SIZE)

def mol_to_morgan_array(mol):
    fp = morgan_generator.GetFingerprint(mol)
    arr = np.zeros((FP_SIZE,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

# Stack one fingerprint vector per molecule.
X = np.vstack(df["mol"].apply(mol_to_morgan_array).to_numpy())

# y is the solubility target.
y = df["target_logS"].to_numpy()

print("Fingerprint matrix:", X.shape)
print("Target vector:", y.shape)

In [ ]:
# Count active fingerprint bits for each molecule.
df["active_bits"] = X.sum(axis=1)

df[["name", "active_bits", "target_logS"]].sort_values("active_bits", ascending=False).head(10)

### Exercise — compare fingerprint settings

Generate two fingerprint matrices using different radius or bit-size settings. Compare their shape and the average number of active bits per molecule.

Guidance: fingerprint settings change how much structural detail is stored. Larger fingerprints may reduce bit collisions, but they also increase feature dimensionality.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Create two Morgan generators with different settings.


    # Convert molecules into two fingerprint matrices.


    # Compare shape and active-bit counts.


    pass
else:
    print("Set RUN_EXERCISE = True after defining two fingerprint settings.")


### Answer — compare fingerprint settings

Reference solution.

In [ ]:
small_generator = GetMorganGenerator(radius=2, fpSize=1024)
large_generator = GetMorganGenerator(radius=3, fpSize=2048)

def mol_to_array_with_generator(mol, generator):
    fp = generator.GetFingerprint(mol)
    arr = np.zeros((fp.GetNumBits(),), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

X_small = np.vstack(df["mol"].apply(lambda mol: mol_to_array_with_generator(mol, small_generator)))
X_large = np.vstack(df["mol"].apply(lambda mol: mol_to_array_with_generator(mol, large_generator)))

pd.DataFrame([
    {"setting": "radius=2, bits=1024", "shape": X_small.shape, "mean_active_bits": X_small.sum(axis=1).mean()},
    {"setting": "radius=3, bits=2048", "shape": X_large.shape, "mean_active_bits": X_large.sum(axis=1).mean()},
])


## 4. Compare molecules by Tanimoto similarity

Tanimoto similarity compares overlap between binary fingerprints. Values close to 1 indicate highly similar fingerprints; values close to 0 indicate little overlap.

Similarity search is useful for checking whether a query molecule has near neighbours in the dataset. It also helps explain why some predictions may be more reliable than others.

In [ ]:
def mol_to_bitvect(mol):
    return morgan_generator.GetFingerprint(mol)

# Store RDKit fingerprint objects for similarity calculations.
df["fingerprint"] = df["mol"].apply(mol_to_bitvect)

# Use the first available molecule as a reference so this works with random public samples.
query_index = 0
query_name = df.loc[query_index, "name"]
query_fp = df.loc[query_index, "fingerprint"]

# Compare the reference molecule against every molecule in the table.
df["similarity_to_reference"] = df["fingerprint"].apply(lambda fp: DataStructs.TanimotoSimilarity(query_fp, fp))

print("Reference molecule:", query_name)
df[["name", "similarity_to_reference", "target_logS"]].sort_values("similarity_to_reference", ascending=False).head(10)

### Exercise — build a query-molecule similarity search

Write a function that accepts a query SMILES string and returns the most similar molecules in the dataset using Tanimoto similarity.

Guidance: this is a useful cheminformatics tool by itself. It also helps judge whether a prediction is supported by similar training molecules.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    def search_similar_molecules(query_smiles, top_n=10):
        # Parse the query SMILES.


        # Convert the query molecule into a fingerprint.


        # Calculate Tanimoto similarity against dataset fingerprints.


        # Return the top_n most similar molecules.


        pass

    # Test the function with one query molecule.

else:
    print("Set RUN_EXERCISE = True after writing search_similar_molecules().")


### Answer — build a query-molecule similarity search

Reference solution.

In [ ]:
def search_similar_molecules(query_smiles, top_n=10):
    query_mol = Chem.MolFromSmiles(query_smiles)
    if query_mol is None:
        raise ValueError("Invalid query SMILES")

    query_fp = mol_to_bitvect(query_mol)
    similarities = df["fingerprint"].apply(lambda fp: DataStructs.TanimotoSimilarity(query_fp, fp))

    result = df[["name", "smiles", "target_logS"]].copy()
    result["similarity"] = similarities
    return result.sort_values("similarity", ascending=False).head(top_n)

search_similar_molecules("CC(=O)OC1=CC=CC=C1C(=O)O", top_n=8)


## 5. Train fingerprint QSAR models

Fingerprint models use the binary feature matrix directly. Tree-based models often work well with sparse binary features, while linear models can provide a simpler baseline.

The model comparison is similar to Project 2, but the molecular representation has changed from descriptors to fingerprints.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

fp_models = {
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
}

rows = []
fitted_models = {}

for model_name, model in fp_models.items():
    # Fit each model on fingerprint vectors.
    model.fit(X_train, y_train)
    fitted_models[model_name] = model

    # Predict the held-out test molecules.
    y_pred = model.predict(X_test)

    rows.append({
        "model": model_name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred),
    })

result_df = pd.DataFrame(rows).sort_values("RMSE")
result_df

### Exercise — run a larger fingerprint model comparison

Create a benchmark with at least three models using the fingerprint matrix. Include one linear model and two tree-based models. Compare MAE, RMSE and R².

Guidance: fingerprints can work with many model types. Comparing simple and non-linear models helps show whether the representation is useful beyond one algorithm.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Define a model dictionary with at least three models.


    # Fit each model on X_train and y_train.


    # Calculate MAE, RMSE and R2 on X_test.


    # Display a sorted comparison table.


    pass
else:
    print("Set RUN_EXERCISE = True after writing the fingerprint benchmark.")


### Answer — run a larger fingerprint model comparison

Reference solution.

In [ ]:
fingerprint_models = {
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=200, random_state=42, n_jobs=-1),
}

fingerprint_rows = []
for model_name, model in fingerprint_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    fingerprint_rows.append({
        "model": model_name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
    })

fingerprint_benchmark_df = pd.DataFrame(fingerprint_rows).sort_values("RMSE")
fingerprint_benchmark_df


## 6. Cross-validation

Cross-validation checks whether fingerprint model performance is stable across different train/validation folds. This is useful because a random split can give an overly optimistic or pessimistic result.

For fingerprints, fold stability can also reflect whether similar molecules are distributed across the dataset.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_rows = []
for model_name, model in fp_models.items():
    # Convert negative MSE scores back into RMSE.
    neg_mse = cross_val_score(model, X, y, cv=cv, scoring="neg_mean_squared_error")
    rmse_scores = np.sqrt(-neg_mse)

    cv_rows.append({
        "model": model_name,
        "CV_RMSE_mean": rmse_scores.mean(),
        "CV_RMSE_std": rmse_scores.std(),
    })

cv_df = pd.DataFrame(cv_rows).sort_values("CV_RMSE_mean")
cv_df

## 7. Error analysis

Error analysis checks which molecules are predicted poorly. With fingerprints, large errors may indicate unusual molecules, poor neighbour coverage or fingerprint collisions.

Drawing error molecules and comparing their nearest neighbours can make fingerprint model behaviour easier to interpret.

In [ ]:
best_model_name = result_df.iloc[0]["model"]
best_model = fitted_models[best_model_name]

# Predict held-out molecules.
y_pred = best_model.predict(X_test)

test_indices = df.index.to_numpy()[train_test_split(np.arange(len(df)), test_size=0.25, random_state=42)[1]]
error_df = df.loc[test_indices, ["name", "smiles", "target_logS"]].copy()
error_df["predicted_logS"] = y_pred
error_df["absolute_error"] = np.abs(error_df["predicted_logS"] - error_df["target_logS"])

error_df.sort_values("absolute_error", ascending=False)

In [ ]:
# Plot true vs predicted solubility.
plt.figure(figsize=(6, 6))
plt.scatter(error_df["target_logS"], error_df["predicted_logS"])

# Perfect predictions would lie on this diagonal line.
min_value = min(error_df["target_logS"].min(), error_df["predicted_logS"].min())
max_value = max(error_df["target_logS"].max(), error_df["predicted_logS"].max())
plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")

plt.xlabel("True logS")
plt.ylabel("Predicted logS")
plt.title(f"Morgan fingerprint QSAR: {best_model_name}")
plt.show()

### Exercise — connect error analysis to similarity

For the largest-error test molecule, find its most similar molecules in the full dataset. Compare their solubility values with the observed and predicted value of the error molecule.

Guidance: a large error is easier to interpret when the molecule's nearest neighbours are visible. If neighbours have very different target values, the prediction problem may be intrinsically difficult.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Identify the largest-error molecule from the error table.


    # Use its SMILES as a query for similarity search.


    # Compare neighbour target_logS values with the error molecule.


    pass
else:
    print("Set RUN_EXERCISE = True after linking error analysis to similarity.")


### Answer — connect error analysis to similarity

Reference solution.

In [ ]:
# Select the largest-error molecule from the existing error table.
largest_error_row = error_df.sort_values("absolute_error", ascending=False).iloc[0]
query_smiles = largest_error_row["smiles"]

nearest_neighbours = search_similar_molecules(query_smiles, top_n=8)

print("Largest-error molecule:")
display(largest_error_row.to_frame().T)

print("Nearest neighbours by Morgan fingerprint similarity:")
display(nearest_neighbours)


## 8. Predict new molecules from fingerprints

New molecules must be parsed, fingerprinted with the same generator and passed into the trained model. The fingerprint size and radius must match the training features.

This part turns the fingerprint QSAR model into a reusable prediction workflow for new SMILES strings.

In [ ]:
def predict_with_fingerprints(smiles_list, model):
    rows = []
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            rows.append({"smiles": smiles, "predicted_logS": np.nan})
            continue

        # Convert the new molecule into the same fingerprint vector.
        fp_array = mol_to_morgan_array(mol).reshape(1, -1)

        rows.append({
            "smiles": smiles,
            "canonical_smiles": Chem.MolToSmiles(mol, canonical=True),
            "predicted_logS": model.predict(fp_array)[0],
        })
    return pd.DataFrame(rows)

new_smiles = ["CCO", "CCCCCCCC", "COc1cc(C=O)ccc1O", "c1ccc2ccccc2c1"]
predict_with_fingerprints(new_smiles, best_model)

### Exercise — predict a custom molecule list

Create your own list of at least six molecules, convert them into Morgan fingerprints and predict logS using the best fingerprint model. Rank the molecules from predicted most soluble to least soluble.

Guidance: this is the reusable prediction path. The same generator and feature length must be used for both training molecules and new molecules.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Create a custom dataframe with name and SMILES columns.


    # Parse and fingerprint each molecule.


    # Predict logS with a fitted fingerprint model.


    # Rank the molecules by predicted logS.


    pass
else:
    print("Set RUN_EXERCISE = True after creating your prediction panel.")


### Answer — predict a custom molecule list

Reference solution.

In [ ]:
custom_queries = pd.DataFrame([
    {"name": "benzyl acetate", "smiles": "CC(=O)OCC1=CC=CC=C1"},
    {"name": "vanillin", "smiles": "COc1cc(C=O)ccc1O"},
    {"name": "limonene", "smiles": "CC1=CCC(CC1)C(=C)C"},
    {"name": "caffeine", "smiles": "Cn1cnc2c1c(=O)n(C)c(=O)n2C"},
    {"name": "glucose", "smiles": "C(C1C(C(C(C(O1)O)O)O)O)O"},
    {"name": "ibuprofen", "smiles": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"},
])

custom_queries["mol"] = custom_queries["smiles"].apply(Chem.MolFromSmiles)
custom_queries = custom_queries[custom_queries["mol"].notnull()].copy()
custom_X = np.vstack(custom_queries["mol"].apply(mol_to_morgan_array))

best_fp_model_name = fingerprint_benchmark_df.iloc[0]["model"]
best_fp_model = fingerprint_models[best_fp_model_name]
best_fp_model.fit(X, y)

custom_queries["predicted_logS"] = best_fp_model.predict(custom_X)
custom_queries[["name", "smiles", "predicted_logS"]].sort_values("predicted_logS", ascending=False)
